# JSON Serialization — Advanced Problems with Solutions

This notebook continues the topic of JSON serialization, but in a **tutorial/problem-solving style**.

Instead of jumping directly to a large serializer, we will build our understanding in small logical steps.

We will repeatedly:

1. look at a concrete problem,
2. predict what Python will do,
3. run a small experiment,
4. explain the result,
5. improve the design,
6. solve a more realistic version of the problem.

The goal is not only to know the `json` module, but to understand the **design decisions** behind reliable serialization.


We will work with the standard library only.

That means all examples can be run in a normal Python installation without third-party packages.


In [2]:

import base64
import json
import math
from dataclasses import dataclass
from datetime import date, datetime, timezone
from decimal import Decimal
from enum import Enum
from fractions import Fraction
from pathlib import Path
from uuid import UUID, uuid4


Before we begin, remember the most important idea:

> JSON does not know anything about Python classes.

JSON only knows a small set of values such as strings, numbers, booleans, arrays, objects and null.

So whenever we serialize a richer Python object, we have to answer a design question:

**How should this Python object be represented using only JSON-compatible values?**


## Problem 1 — Is a successful `dumps()` enough?

Suppose we have the following Python object:


In [3]:

data = {
    "name": "Alice",
    "scores": (10, 20, 30),
    "active": True
}


At first glance this looks very JSON-friendly.

Let us serialize it.


In [4]:

text = json.dumps(data)
text


'{"name": "Alice", "scores": [10, 20, 30], "active": true}'

The serialization succeeds.

So does that mean the object can be perfectly reconstructed?


In [5]:

restored = json.loads(text)

print(restored)
print(type(restored["scores"]))


{'name': 'Alice', 'scores': [10, 20, 30], 'active': True}
<class 'list'>


The tuple became a list.

This is our first advanced lesson:

> Successful serialization does **not** imply lossless round-tripping.

The JSON format has arrays, but it has no separate tuple type.


Let us compare the original object with the restored object.


In [6]:

print(data == restored)


False


The values look similar, but Python considers a tuple and a list different.

So this identity is **not always true**:

```python
obj == json.loads(json.dumps(obj))
```

It only works when the original object uses types whose meaning is preserved by the JSON mapping.


### Solution strategy

If the distinction between list and tuple matters, we need to store some extra information.

One simple representation could be:


In [7]:

tagged_data = {
    "name": "Alice",
    "scores": {
        "__type__": "tuple",
        "items": [10, 20, 30]
    },
    "active": True
}

print(json.dumps(tagged_data, indent=2))


{
  "name": "Alice",
  "scores": {
    "__type__": "tuple",
    "items": [
      10,
      20,
      30
    ]
  },
  "active": true
}


Now the JSON contains enough information for us to reconstruct the tuple intentionally.

We will return to tagged representations later.


## Problem 2 — Dictionary keys can silently change

Python dictionaries can use many hashable objects as keys.

JSON objects are different:

> JSON object keys must be strings.

Let us see what happens with integer keys.


In [8]:

d = {
    1: "one",
    2: "two",
    3: "three"
}

text = json.dumps(d)
text


'{"1": "one", "2": "two", "3": "three"}'

Notice that the JSON keys are now strings.

Let us deserialize the object.


In [9]:

restored = json.loads(text)

print(d)
print(restored)
print(d == restored)


{1: 'one', 2: 'two', 3: 'three'}
{'1': 'one', '2': 'two', '3': 'three'}
False


The values survived, but the key types did not.

This can be especially dangerous because Python does not raise an error here.

The conversion happens silently.


### A safer representation

If key type matters, an object may not be the right JSON structure.

Instead, we can represent key/value pairs as an array of records:


In [10]:

safe_representation = {
    "entries": [
        {"key": 1, "value": "one"},
        {"key": 2, "value": "two"},
        {"key": 3, "value": "three"},
    ]
}

print(json.dumps(safe_representation, indent=2))


{
  "entries": [
    {
      "key": 1,
      "value": "one"
    },
    {
      "key": 2,
      "value": "two"
    },
    {
      "key": 3,
      "value": "three"
    }
  ]
}


Now the integer key is stored as a JSON number inside a normal record.

That preserves its intended type.


## Problem 3 — Non-string dictionary keys that are even more surprising

What happens with boolean and `None` keys?


In [11]:

strange_keys = {
    True: "yes",
    False: "no",
    None: "missing"
}

text = json.dumps(strange_keys)
text


'{"true": "yes", "false": "no", "null": "missing"}'

Again, the keys are converted to strings.

This is another reason not to assume that a dictionary can always round-trip through JSON.


There is another subtle Python issue here.

Remember that:

```python
True == 1
False == 0
```

and they also have equal hashes.

So even before JSON is involved, some Python dictionary keys can collide.


In [12]:

collision = {
    True: "boolean",
    1: "integer"
}

collision


{True: 'integer'}

Only one entry survives.

This is not a JSON problem — it is normal Python dictionary behavior.

But serialization work often exposes these representation issues, so it is important to reason about the data model before encoding it.


## Problem 4 — Strict JSON and special floating-point values

Python floats can represent:

- `nan`
- positive infinity
- negative infinity

Let us see what `json.dumps()` does by default.


In [13]:

special_numbers = {
    "nan": float("nan"),
    "positive_infinity": float("inf"),
    "negative_infinity": float("-inf")
}

print(json.dumps(special_numbers))


{"nan": NaN, "positive_infinity": Infinity, "negative_infinity": -Infinity}


Python emits:

- `NaN`
- `Infinity`
- `-Infinity`

However, these are not part of strict JSON.

Some other systems may reject this output.


The `json` module gives us an option to enforce stricter behavior:


In [14]:

try:
    json.dumps(special_numbers, allow_nan=False)
except ValueError as ex:
    print(type(ex).__name__)
    print(ex)


ValueError
Out of range float values are not JSON compliant: nan


This is often the better choice for APIs, files exchanged with other languages, and long-lived data formats.

A useful best practice is:

```python
json.dumps(data, allow_nan=False)
```

when interoperability matters.


### A small validation helper

Instead of discovering the problem during serialization, we can also validate our object first.


In [15]:

def validate_finite_numbers(value, path="$"):
    if isinstance(value, float):
        if not math.isfinite(value):
            raise ValueError(f"Non-finite float at {path}: {value!r}")
        return

    if isinstance(value, dict):
        for key, item in value.items():
            validate_finite_numbers(item, f"{path}.{key}")
        return

    if isinstance(value, (list, tuple)):
        for index, item in enumerate(value):
            validate_finite_numbers(item, f"{path}[{index}]")


In [16]:

try:
    validate_finite_numbers({
        "measurements": [1.2, 3.4, float("nan")]
    })
except ValueError as ex:
    print(ex)


Non-finite float at $.measurements[2]: nan


Notice that the error tells us exactly where the invalid value occurred.

For larger documents, path-aware validation errors are much easier to debug.


## Problem 5 — Why `Decimal` needs special treatment

Consider a financial value:


In [17]:

price = Decimal("0.10")
price


Decimal('0.10')

Let us try to serialize it directly.


In [18]:

try:
    json.dumps({"price": price})
except TypeError as ex:
    print(ex)


Object of type Decimal is not JSON serializable


`Decimal` is not one of the types supported automatically by the standard JSON encoder.

We have to decide how to represent it.


A tempting solution is to convert it to `float`.


In [19]:

float_price = float(price)

print(float_price)
print(json.dumps({"price": float_price}))


0.1
{"price": 0.1}


That may look fine for `0.10`.

But floating-point numbers are binary approximations.

For financial and high-precision values, converting to float can lose information.


In [20]:

precise = Decimal("1234567890.12345678901234567890")

print("Decimal:", precise)
print("Float:  ", float(precise))


Decimal: 1234567890.12345678901234567890
Float:   1234567890.1234567


A safer approach is to serialize the decimal digits as a string.


In [21]:

decimal_document = {
    "price": {
        "__type__": "decimal",
        "value": str(precise)
    }
}

print(json.dumps(decimal_document, indent=2))


{
  "price": {
    "__type__": "decimal",
    "value": "1234567890.12345678901234567890"
  }
}


The JSON string is exact.

Now we can deliberately reconstruct the `Decimal`.


In [22]:

text = json.dumps(decimal_document)
raw = json.loads(text)

restored_price = Decimal(raw["price"]["value"])

print(restored_price)
print(restored_price == precise)


1234567890.12345678901234567890
True


## Problem 6 — Using the `default` argument

Manually converting every `Decimal` before every call to `json.dumps()` would quickly become annoying.

Fortunately, `json.dumps()` accepts a `default` callable.

That callable is used when the encoder encounters an unsupported object.


In [23]:

def decimal_default(obj):
    if isinstance(obj, Decimal):
        return {
            "__type__": "decimal",
            "value": str(obj)
        }

    raise TypeError(
        f"Object of type {type(obj).__name__} is not JSON serializable"
    )


Let us use it.


In [24]:

data = {
    "subtotal": Decimal("19.99"),
    "tax": Decimal("1.60"),
    "total": Decimal("21.59")
}

text = json.dumps(data, default=decimal_default, indent=2)
print(text)


{
  "subtotal": {
    "__type__": "decimal",
    "value": "19.99"
  },
  "tax": {
    "__type__": "decimal",
    "value": "1.60"
  },
  "total": {
    "__type__": "decimal",
    "value": "21.59"
  }
}


This is much more convenient.

The encoder walks through the object recursively.

Whenever it finds a normal JSON-compatible value, it handles it automatically.

Whenever it finds a `Decimal`, it calls our function.


### Why should the function raise `TypeError`?

Suppose the object contains an unsupported type we did not plan for.

Silently returning something arbitrary would hide a bug.

A serializer should fail loudly for types it does not understand.


## Problem 7 — Decoding tagged objects with `object_hook`

Encoding is only half the story.

We also want to restore tagged decimal objects automatically.

The `json.loads()` function provides `object_hook`.

The hook receives each JSON object after its contents have been decoded into a dictionary.


In [25]:

def decimal_object_hook(obj):
    if obj.get("__type__") == "decimal":
        return Decimal(obj["value"])

    return obj


Now we can deserialize our previous document.


In [26]:

restored = json.loads(
    text,
    object_hook=decimal_object_hook
)

print(restored)
print(type(restored["total"]))


{'subtotal': Decimal('19.99'), 'tax': Decimal('1.60'), 'total': Decimal('21.59')}
<class 'decimal.Decimal'>


The `total` value is a real `Decimal` again.


In [27]:

assert restored["total"] == Decimal("21.59")


This gives us a complete round trip:

```text
Decimal
   ↓
tagged JSON object
   ↓
JSON text
   ↓
tagged Python dictionary
   ↓
Decimal
```


## Problem 8 — Supporting several non-JSON-native types

A realistic application may use many types:

- `Decimal`
- `Fraction`
- `complex`
- `set`
- `datetime`
- `UUID`

Let us build a multi-type serializer.


In [28]:

def advanced_default(obj):
    if isinstance(obj, Decimal):
        return {
            "__type__": "decimal",
            "value": str(obj)
        }

    if isinstance(obj, Fraction):
        return {
            "__type__": "fraction",
            "numerator": obj.numerator,
            "denominator": obj.denominator
        }

    if isinstance(obj, complex):
        return {
            "__type__": "complex",
            "real": obj.real,
            "imag": obj.imag
        }

    if isinstance(obj, set):
        return {
            "__type__": "set",
            "items": sorted(obj)
        }

    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat()
        }

    if isinstance(obj, UUID):
        return {
            "__type__": "uuid",
            "value": str(obj)
        }

    raise TypeError(
        f"Unsupported type: {type(obj).__name__}"
    )


Let us create an object containing all of these types.


In [29]:

data = {
    "price": Decimal("99.95"),
    "ratio": Fraction(2, 3),
    "signal": 3 + 4j,
    "tags": {"python", "json", "serialization"},
    "created_at": datetime(
        2026, 8, 7, 12, 30,
        tzinfo=timezone.utc
    ),
    "request_id": uuid4()
}


In [30]:

text = json.dumps(
    data,
    default=advanced_default,
    indent=2,
    sort_keys=True
)

print(text)


{
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T12:30:00+00:00"
  },
  "price": {
    "__type__": "decimal",
    "value": "99.95"
  },
  "ratio": {
    "__type__": "fraction",
    "denominator": 3,
    "numerator": 2
  },
  "request_id": {
    "__type__": "uuid",
    "value": "39c43e30-434f-4a95-9789-7c712eded3d9"
  },
  "signal": {
    "__type__": "complex",
    "imag": 4.0,
    "real": 3.0
  },
  "tags": {
    "__type__": "set",
    "items": [
      "json",
      "python",
      "serialization"
    ]
  }
}


Each unsupported Python value is now represented by an ordinary JSON object.

The important part is that the representation is **explicit**.

We are not merely converting everything to strings and hoping to guess the type later.


## Problem 9 — Building the matching decoder

Now we need the reverse transformation.


In [31]:

def advanced_object_hook(obj):
    type_name = obj.get("__type__")

    if type_name == "decimal":
        return Decimal(obj["value"])

    if type_name == "fraction":
        return Fraction(
            obj["numerator"],
            obj["denominator"]
        )

    if type_name == "complex":
        return complex(
            obj["real"],
            obj["imag"]
        )

    if type_name == "set":
        return set(obj["items"])

    if type_name == "datetime":
        return datetime.fromisoformat(
            obj["value"]
        )

    if type_name == "uuid":
        return UUID(obj["value"])

    return obj


Let us restore the object.


In [32]:

restored = json.loads(
    text,
    object_hook=advanced_object_hook
)


In [33]:

for key, value in restored.items():
    print(
        key,
        "=>",
        repr(value),
        "=>",
        type(value).__name__
    )


created_at => datetime.datetime(2026, 8, 7, 12, 30, tzinfo=datetime.timezone.utc) => datetime
price => Decimal('99.95') => Decimal
ratio => Fraction(2, 3) => Fraction
request_id => UUID('39c43e30-434f-4a95-9789-7c712eded3d9') => UUID
signal => (3+4j) => complex
tags => {'python', 'serialization', 'json'} => set


And we can test the important fields.


In [34]:

assert restored["price"] == data["price"]
assert restored["ratio"] == data["ratio"]
assert restored["signal"] == data["signal"]
assert restored["tags"] == data["tags"]
assert restored["created_at"] == data["created_at"]
assert restored["request_id"] == data["request_id"]

print("Round trip succeeded.")


Round trip succeeded.


## Problem 10 — A trap: why `default` does not preserve tuples

Earlier, we saw that tuples become JSON arrays.

You might think we can simply add this to `advanced_default`:

```python
if isinstance(obj, tuple):
    ...
```

But there is a problem.

The standard encoder already knows how to serialize tuples.

It converts them to JSON arrays **before** `default` is needed.


In [35]:

point = (10, 20)

text = json.dumps(
    {"point": point},
    default=advanced_default
)

print(text)


{"point": [10, 20]}


Our `default` function never gets a chance to tag the tuple.

When we load the JSON:


In [36]:

restored = json.loads(text)

print(restored)
print(type(restored["point"]))


{'point': [10, 20]}
<class 'list'>


So if tuple preservation matters, we need a different technique.

One solution is to recursively transform the Python object **before** calling `json.dumps()`.


## Problem 11 — Recursive preprocessing

Let us write a function that converts a Python object tree into a JSON-compatible object tree.

This gives us complete control before the standard JSON encoder sees the data.


In [37]:

def prepare_for_json(value):
    if isinstance(value, tuple):
        return {
            "__type__": "tuple",
            "items": [
                prepare_for_json(item)
                for item in value
            ]
        }

    if isinstance(value, list):
        return [
            prepare_for_json(item)
            for item in value
        ]

    if isinstance(value, dict):
        return {
            str(key): prepare_for_json(item)
            for key, item in value.items()
        }

    if isinstance(value, Decimal):
        return {
            "__type__": "decimal",
            "value": str(value)
        }

    if isinstance(value, Fraction):
        return {
            "__type__": "fraction",
            "numerator": value.numerator,
            "denominator": value.denominator
        }

    if isinstance(value, set):
        return {
            "__type__": "set",
            "items": [
                prepare_for_json(item)
                for item in sorted(value)
            ]
        }

    if isinstance(value, datetime):
        return {
            "__type__": "datetime",
            "value": value.isoformat()
        }

    if isinstance(value, UUID):
        return {
            "__type__": "uuid",
            "value": str(value)
        }

    if value is None:
        return value

    if isinstance(value, (str, int, float, bool)):
        return value

    raise TypeError(
        f"Unsupported type: {type(value).__name__}"
    )


Now tuples can be detected before the built-in encoder sees them.


In [38]:

data = {
    "point": (10, 20),
    "nested": [
        (1, 2),
        (3, 4)
    ],
    "amount": Decimal("10.50")
}

prepared = prepare_for_json(data)

print(json.dumps(prepared, indent=2))


{
  "point": {
    "__type__": "tuple",
    "items": [
      10,
      20
    ]
  },
  "nested": [
    {
      "__type__": "tuple",
      "items": [
        1,
        2
      ]
    },
    {
      "__type__": "tuple",
      "items": [
        3,
        4
      ]
    }
  ],
  "amount": {
    "__type__": "decimal",
    "value": "10.50"
  }
}


Let us add tuple support to our decoder.


In [39]:

def full_object_hook(obj):
    type_name = obj.get("__type__")

    if type_name == "tuple":
        return tuple(obj["items"])

    if type_name == "decimal":
        return Decimal(obj["value"])

    if type_name == "fraction":
        return Fraction(
            obj["numerator"],
            obj["denominator"]
        )

    if type_name == "set":
        return set(obj["items"])

    if type_name == "datetime":
        return datetime.fromisoformat(
            obj["value"]
        )

    if type_name == "uuid":
        return UUID(obj["value"])

    return obj


In [40]:

text = json.dumps(
    prepare_for_json(data)
)

restored = json.loads(
    text,
    object_hook=full_object_hook
)

print(restored)
print(type(restored["point"]))
print(type(restored["nested"][0]))


{'point': (10, 20), 'nested': [(1, 2), (3, 4)], 'amount': Decimal('10.50')}
<class 'tuple'>
<class 'tuple'>


Now tuple information survives the round trip.


## Problem 12 — Parsing external JSON numbers precisely

So far we controlled the serialization format ourselves.

But what if we receive JSON from an external system?

Consider this JSON:


In [41]:

external_json = '''
{
    "price": 0.1,
    "tax": 0.2
}
'''


By default, JSON decimal numbers are parsed as Python floats.


In [42]:

normal = json.loads(external_json)

print(normal)
print(type(normal["price"]))
print(normal["price"] + normal["tax"])


{'price': 0.1, 'tax': 0.2}
<class 'float'>
0.30000000000000004


The result is the classic floating-point approximation:


In [43]:

normal["price"] + normal["tax"] == 0.3


False

The decoder lets us choose how floating-point JSON numbers should be parsed.

We can use `Decimal`.


In [44]:

precise = json.loads(
    external_json,
    parse_float=Decimal
)

print(precise)
print(type(precise["price"]))
print(precise["price"] + precise["tax"])


{'price': Decimal('0.1'), 'tax': Decimal('0.2')}
<class 'decimal.Decimal'>
0.3


In [45]:

assert precise["price"] + precise["tax"] == Decimal("0.3")


This is an extremely useful technique for financial JSON received from external APIs.

Notice the difference:

- `default=...` affects **encoding**
- `parse_float=...` affects **decoding**


## Problem 13 — Duplicate keys in JSON objects

Consider the following JSON string:


In [46]:

duplicate_json = '''
{
    "role": "user",
    "role": "admin"
}
'''


What should happen?

Let us see Python's default behavior.


In [47]:

json.loads(duplicate_json)


{'role': 'admin'}

The second value silently replaces the first.

For ordinary data this may be acceptable.

For security-sensitive configuration, authorization rules, or signed data, silently accepting duplicate keys may be risky.


The decoder can expose object entries as a sequence of pairs using `object_pairs_hook`.


In [48]:

def reject_duplicate_keys(pairs):
    result = {}

    for key, value in pairs:
        if key in result:
            raise ValueError(
                f"Duplicate JSON key: {key!r}"
            )

        result[key] = value

    return result


In [49]:

try:
    json.loads(
        duplicate_json,
        object_pairs_hook=reject_duplicate_keys
    )
except ValueError as ex:
    print(ex)


Duplicate JSON key: 'role'


Now the ambiguity is rejected explicitly.


## Problem 14 — Serializing a domain object

Let us move from individual types to a small application model.

Suppose we have an invoice.


In [50]:

@dataclass(frozen=True)
class Invoice:
    invoice_id: UUID
    customer: str
    amount: Decimal
    issued_at: datetime


In [51]:

invoice = Invoice(
    invoice_id=uuid4(),
    customer="Grace Hopper",
    amount=Decimal("1250.75"),
    issued_at=datetime(
        2026, 8, 7, 13, 0,
        tzinfo=timezone.utc
    )
)

invoice


Invoice(invoice_id=UUID('a34b5d57-a9f2-443d-9925-9cbe290d2f60'), customer='Grace Hopper', amount=Decimal('1250.75'), issued_at=datetime.datetime(2026, 8, 7, 13, 0, tzinfo=datetime.timezone.utc))

Trying to serialize the dataclass directly will fail.


In [52]:

try:
    json.dumps(invoice)
except TypeError as ex:
    print(ex)


Object of type Invoice is not JSON serializable


A dataclass is still a Python class.

JSON does not know its fields automatically.


One possible approach is `vars()` or `dataclasses.asdict()`.

But even after converting the dataclass to a dictionary, some field values are still not JSON-native.


In [53]:

invoice_dict = {
    "invoice_id": invoice.invoice_id,
    "customer": invoice.customer,
    "amount": invoice.amount,
    "issued_at": invoice.issued_at
}

invoice_dict


{'invoice_id': UUID('a34b5d57-a9f2-443d-9925-9cbe290d2f60'),
 'customer': 'Grace Hopper',
 'amount': Decimal('1250.75'),
 'issued_at': datetime.datetime(2026, 8, 7, 13, 0, tzinfo=datetime.timezone.utc)}

So we should design an explicit document representation.


In [54]:

def invoice_to_document(invoice):
    return {
        "invoice_id": str(invoice.invoice_id),
        "customer": invoice.customer,
        "amount": str(invoice.amount),
        "issued_at": invoice.issued_at.isoformat()
    }


In [55]:

document = invoice_to_document(invoice)

print(json.dumps(document, indent=2))


{
  "invoice_id": "a34b5d57-a9f2-443d-9925-9cbe290d2f60",
  "customer": "Grace Hopper",
  "amount": "1250.75",
  "issued_at": "2026-08-07T13:00:00+00:00"
}


And now the inverse function:


In [56]:

def invoice_from_document(document):
    return Invoice(
        invoice_id=UUID(
            document["invoice_id"]
        ),
        customer=document["customer"],
        amount=Decimal(
            document["amount"]
        ),
        issued_at=datetime.fromisoformat(
            document["issued_at"]
        )
    )


In [57]:

text = json.dumps(
    invoice_to_document(invoice)
)

restored_invoice = invoice_from_document(
    json.loads(text)
)

print(restored_invoice)
print(restored_invoice == invoice)


Invoice(invoice_id=UUID('a34b5d57-a9f2-443d-9925-9cbe290d2f60'), customer='Grace Hopper', amount=Decimal('1250.75'), issued_at=datetime.datetime(2026, 8, 7, 13, 0, tzinfo=datetime.timezone.utc))
True


This pattern is extremely important in real applications:

> Domain object ↔ JSON document representation

The application class and the wire/storage format do not need to be identical.


## Problem 15 — Parsing is not validation

Suppose the following JSON is syntactically valid:


In [58]:

bad_invoice_json = '''
{
    "invoice_id": "not-a-uuid",
    "customer": "",
    "amount": "-100.00",
    "issued_at": "2026-08-07T13:00:00"
}
'''


`json.loads()` can parse it perfectly.


In [59]:

bad_document = json.loads(
    bad_invoice_json
)

bad_document


{'invoice_id': 'not-a-uuid',
 'customer': '',
 'amount': '-100.00',
 'issued_at': '2026-08-07T13:00:00'}

But the document may violate our application rules:

- the UUID is invalid,
- the customer is empty,
- the amount is negative,
- the datetime is missing timezone information.

JSON syntax validation is therefore not enough.


Let us write a defensive decoder.


In [60]:

def validated_invoice_from_document(document):
    customer = document.get("customer")

    if not isinstance(customer, str):
        raise ValueError(
            "customer must be a string"
        )

    if not customer.strip():
        raise ValueError(
            "customer must not be empty"
        )

    try:
        invoice_id = UUID(
            document["invoice_id"]
        )
    except (KeyError, ValueError) as ex:
        raise ValueError(
            "invoice_id must be a valid UUID"
        ) from ex

    try:
        amount = Decimal(
            document["amount"]
        )
    except Exception as ex:
        raise ValueError(
            "amount must be a valid decimal"
        ) from ex

    if amount < 0:
        raise ValueError(
            "amount must not be negative"
        )

    try:
        issued_at = datetime.fromisoformat(
            document["issued_at"]
        )
    except (KeyError, ValueError) as ex:
        raise ValueError(
            "issued_at must be ISO-8601 datetime text"
        ) from ex

    if issued_at.tzinfo is None:
        raise ValueError(
            "issued_at must include timezone information"
        )

    return Invoice(
        invoice_id=invoice_id,
        customer=customer,
        amount=amount,
        issued_at=issued_at
    )


In [61]:

try:
    validated_invoice_from_document(
        bad_document
    )
except ValueError as ex:
    print(ex)


customer must not be empty


The decoder now rejects values that are valid JSON but invalid application data.


## Problem 16 — `date` and `datetime` ordering in type checks

There is a subtle Python inheritance relationship:


In [62]:

today = date(2026, 8, 7)
now = datetime(
    2026, 8, 7, 15, 0,
    tzinfo=timezone.utc
)

print(isinstance(today, date))
print(isinstance(now, datetime))
print(isinstance(now, date))


True
True
True


A `datetime` is also considered a `date`.

This matters in custom serializers.


Consider this incorrect ordering:


In [63]:

def bad_date_serializer(obj):
    if isinstance(obj, date):
        return {
            "__type__": "date",
            "value": obj.isoformat()
        }

    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat()
        }

    raise TypeError


The `datetime` branch can never be reached for a datetime object, because the earlier `date` check already matches it.


In [64]:

print(
    json.dumps(
        {"when": now},
        default=bad_date_serializer,
        indent=2
    )
)


{
  "when": {
    "__type__": "date",
    "value": "2026-08-07T15:00:00+00:00"
  }
}


The solution is to test the more specific type first.


In [65]:

def good_date_serializer(obj):
    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat()
        }

    if isinstance(obj, date):
        return {
            "__type__": "date",
            "value": obj.isoformat()
        }

    raise TypeError


In [66]:

print(
    json.dumps(
        {
            "when": now,
            "day": today
        },
        default=good_date_serializer,
        indent=2
    )
)


{
  "when": {
    "__type__": "datetime",
    "value": "2026-08-07T15:00:00+00:00"
  },
  "day": {
    "__type__": "date",
    "value": "2026-08-07"
  }
}


This is a general Python design lesson:

> When related types overlap through inheritance, test the more specific type before the more general type.


## Problem 17 — Binary data cannot be placed directly in JSON

JSON is a text format.

What happens with `bytes`?


In [67]:

binary_data = b"\x00\x01hello\xff"

try:
    json.dumps(
        {"payload": binary_data}
    )
except TypeError as ex:
    print(ex)


Object of type bytes is not JSON serializable


We need a textual representation.

A common choice is Base64.


In [68]:

encoded_bytes = base64.b64encode(
    binary_data
).decode("ascii")

encoded_bytes


'AAFoZWxsb/8='

Now it can be stored in JSON.


In [69]:

document = {
    "payload": {
        "__type__": "bytes",
        "encoding": "base64",
        "value": encoded_bytes
    }
}

print(json.dumps(document, indent=2))


{
  "payload": {
    "__type__": "bytes",
    "encoding": "base64",
    "value": "AAFoZWxsb/8="
  }
}


And decoding is the reverse operation.


In [70]:

raw = json.loads(
    json.dumps(document)
)

restored_bytes = base64.b64decode(
    raw["payload"]["value"].encode("ascii")
)

print(restored_bytes)
print(restored_bytes == binary_data)


b'\x00\x01hello\xff'
True


Base64 increases the size of the data, but it is widely supported and safe for text-based transport.


## Problem 18 — Enums

Enums are another common application type.


In [71]:

class Status(Enum):
    PENDING = "pending"
    PAID = "paid"
    CANCELLED = "cancelled"


In [72]:

status = Status.PAID
status


<Status.PAID: 'paid'>

The enum object itself is not directly JSON serializable.


In [73]:

try:
    json.dumps({"status": status})
except TypeError as ex:
    print(ex)


Object of type Status is not JSON serializable


For an API or storage document, the enum value is often the most useful representation.


In [74]:

document = {
    "status": status.value
}

print(json.dumps(document))


{"status": "paid"}


When loading, the Enum constructor can validate the value.


In [75]:

raw = json.loads(
    '{"status": "paid"}'
)

restored_status = Status(
    raw["status"]
)

print(restored_status)


Status.PAID


And an unknown value is rejected.


In [76]:

try:
    Status("refunded")
except ValueError as ex:
    print(ex)


'refunded' is not a valid Status


This is a nice example of using domain types to validate decoded JSON values.


## Problem 19 — A custom `JSONEncoder` class

The `default=` callback is simple and often sufficient.

For larger codebases, you may prefer a custom encoder class.


In [77]:

class ApplicationJSONEncoder(
    json.JSONEncoder
):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return {
                "__type__": "decimal",
                "value": str(obj)
            }

        if isinstance(obj, UUID):
            return {
                "__type__": "uuid",
                "value": str(obj)
            }

        if isinstance(obj, datetime):
            return {
                "__type__": "datetime",
                "value": obj.isoformat()
            }

        if isinstance(obj, set):
            return {
                "__type__": "set",
                "items": sorted(obj)
            }

        return super().default(obj)


The final line is important:

```python
return super().default(obj)
```

It preserves the normal error behavior for types the encoder does not support.


In [78]:

data = {
    "amount": Decimal("42.50"),
    "request_id": uuid4(),
    "created_at": datetime(
        2026, 8, 7, 16, 0,
        tzinfo=timezone.utc
    ),
    "features": {"json", "python"}
}

print(
    json.dumps(
        data,
        cls=ApplicationJSONEncoder,
        indent=2
    )
)


{
  "amount": {
    "__type__": "decimal",
    "value": "42.50"
  },
  "request_id": {
    "__type__": "uuid",
    "value": "b348fa42-0f9e-4be8-a0ce-5525bf34ac25"
  },
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T16:00:00+00:00"
  },
  "features": {
    "__type__": "set",
    "items": [
      "json",
      "python"
    ]
  }
}


## Problem 20 — Deterministic JSON output

Two Python dictionaries may contain the same logical content in different insertion orders.


In [79]:

a = {
    "z": 3,
    "a": 1,
    "m": 2
}

b = {
    "m": 2,
    "z": 3,
    "a": 1
}

print(json.dumps(a))
print(json.dumps(b))


{"z": 3, "a": 1, "m": 2}
{"m": 2, "z": 3, "a": 1}


The output strings may differ because the insertion order differs.

For caching, reproducible builds, testing or hashing, we may want deterministic output.


In [80]:

def deterministic_json(value):
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False
    )


In [81]:

text_a = deterministic_json(a)
text_b = deterministic_json(b)

print(text_a)
print(text_b)
print(text_a == text_b)


{"a":1,"m":2,"z":3}
{"a":1,"m":2,"z":3}
True


The options do different jobs:

- `sort_keys=True` sorts object keys,
- `separators=(",", ":")` removes unnecessary whitespace,
- `ensure_ascii=False` keeps Unicode readable,
- `allow_nan=False` prevents non-standard numeric values.


This produces useful deterministic output for ordinary application data.

But for cryptographic protocols, always follow the exact canonicalization specification required by that protocol.


## Problem 21 — Unicode and `ensure_ascii`

JSON is Unicode text.

Let us serialize some non-ASCII text.


In [82]:

message = {
    "english": "hello",
    "bulgarian": "Здравей",
    "japanese": "こんにちは"
}


In [83]:

print(
    json.dumps(
        message,
        indent=2
    )
)


{
  "english": "hello",
  "bulgarian": "\u0417\u0434\u0440\u0430\u0432\u0435\u0439",
  "japanese": "\u3053\u3093\u306b\u3061\u306f"
}


By default, non-ASCII characters are escaped.


In [84]:

print(
    json.dumps(
        message,
        indent=2,
        ensure_ascii=False
    )
)


{
  "english": "hello",
  "bulgarian": "Здравей",
  "japanese": "こんにちは"
}


Both forms represent the same data.

For human-readable UTF-8 files, `ensure_ascii=False` is often more convenient.


## Problem 22 — `dump` and `load` with files

So far we mostly used:

- `dumps()` — serialize to a string
- `loads()` — deserialize from a string

For files we usually use:

- `dump()` — write JSON to a file-like object
- `load()` — read JSON from a file-like object


In [85]:

config = {
    "application": "json-demo",
    "debug": False,
    "limits": {
        "max_items": 500,
        "timeout": 2.5
    }
}

config_path = Path(
    "tutorial_config.json"
)


Let us save it.


In [86]:

with config_path.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )


And read it back.


In [87]:

with config_path.open(
    "r",
    encoding="utf-8"
) as f:
    restored_config = json.load(f)

restored_config


{'application': 'json-demo',
 'debug': False,
 'limits': {'max_items': 500, 'timeout': 2.5}}

In [88]:

assert restored_config == config


Explicit UTF-8 encoding is a good habit for JSON files.


## Problem 23 — Reporting useful JSON syntax errors

Malformed JSON raises `JSONDecodeError`.


In [89]:

bad_json = '''
{
    "name": "Alice",
    "age": 30,
}
'''


The trailing comma is invalid JSON.

Let us inspect the exception.


In [90]:

try:
    json.loads(bad_json)
except json.JSONDecodeError as ex:
    print("Message:", ex.msg)
    print("Line:", ex.lineno)
    print("Column:", ex.colno)
    print("Character:", ex.pos)


Message: Illegal trailing comma before end of object
Line: 4
Column: 14
Character: 37


The line and column are useful when building configuration readers or developer tools.

We can wrap the error with additional context.


In [91]:

def load_json_text(text, source="<string>"):
    try:
        return json.loads(text)
    except json.JSONDecodeError as ex:
        raise ValueError(
            f"Invalid JSON in {source}: "
            f"line {ex.lineno}, "
            f"column {ex.colno}: "
            f"{ex.msg}"
        ) from ex


In [92]:

try:
    load_json_text(
        bad_json,
        source="settings.json"
    )
except ValueError as ex:
    print(ex)


Invalid JSON in settings.json: line 4, column 14: Illegal trailing comma before end of object


## Problem 24 — JSON Lines for event streams

A large JSON array looks like this:


In [93]:

events = [
    {
        "event_id": 1,
        "type": "login"
    },
    {
        "event_id": 2,
        "type": "purchase"
    },
    {
        "event_id": 3,
        "type": "logout"
    }
]

print(json.dumps(events, indent=2))


[
  {
    "event_id": 1,
    "type": "login"
  },
  {
    "event_id": 2,
    "type": "purchase"
  },
  {
    "event_id": 3,
    "type": "logout"
  }
]


This is fine for small datasets.

But for logs and streams, a useful alternative is **JSON Lines**.

Each line is a complete JSON document:


In [94]:

jsonl_path = Path(
    "events_tutorial.jsonl"
)

with jsonl_path.open(
    "w",
    encoding="utf-8"
) as f:
    for event in events:
        json.dump(event, f)
        f.write("\n")


In [95]:

print(
    jsonl_path.read_text(
        encoding="utf-8"
    )
)


{"event_id": 1, "type": "login"}
{"event_id": 2, "type": "purchase"}
{"event_id": 3, "type": "logout"}



The advantage is that we can process one record at a time.


In [96]:

with jsonl_path.open(
    "r",
    encoding="utf-8"
) as f:
    for line_number, line in enumerate(
        f,
        start=1
    ):
        if not line.strip():
            continue

        event = json.loads(line)

        print(
            line_number,
            event
        )


1 {'event_id': 1, 'type': 'login'}
2 {'event_id': 2, 'type': 'purchase'}
3 {'event_id': 3, 'type': 'logout'}


This avoids loading a giant array into memory all at once.


## Problem 25 — Circular references

JSON naturally represents trees.

Python object graphs can contain cycles.

Let us create one.


In [97]:

parent = {
    "name": "parent"
}

child = {
    "name": "child",
    "parent": parent
}

parent["child"] = child


Now `parent` refers to `child`, and `child` refers back to `parent`.

There is no finite nested JSON tree for this structure.


In [98]:

try:
    json.dumps(parent)
except ValueError as ex:
    print(ex)


Circular reference detected


The correct solution is usually not to disable circular checking.

Instead, redesign the serialized representation.

For example, use object IDs.


In [99]:

graph_document = {
    "root": "P1",
    "nodes": {
        "P1": {
            "name": "parent",
            "child_id": "C1"
        },
        "C1": {
            "name": "child",
            "parent_id": "P1"
        }
    }
}

print(
    json.dumps(
        graph_document,
        indent=2
    )
)


{
  "root": "P1",
  "nodes": {
    "P1": {
      "name": "parent",
      "child_id": "C1"
    },
    "C1": {
      "name": "child",
      "parent_id": "P1"
    }
  }
}


The JSON is now a finite document.

Relationships are represented explicitly instead of through recursive object references.


## Problem 26 — Schema versioning

Serialization formats often live longer than expected.

Suppose version 1 of a customer document looked like this:


In [100]:

customer_v1 = {
    "schema_version": 1,
    "name": "Ada Lovelace",
    "email": "ada@example.com"
}


Later, version 2 changes the shape:


In [101]:

customer_v2 = {
    "schema_version": 2,
    "profile": {
        "full_name": "Ada Lovelace",
        "emails": [
            "ada@example.com"
        ]
    }
}


Old files may still exist.

A migration function can convert older representations into the current one.


In [102]:

def migrate_customer(document):
    version = document.get(
        "schema_version"
    )

    if version == 2:
        return document

    if version == 1:
        return {
            "schema_version": 2,
            "profile": {
                "full_name": document["name"],
                "emails": (
                    [document["email"]]
                    if document["email"]
                    else []
                )
            }
        }

    raise ValueError(
        f"Unsupported schema version: {version!r}"
    )


In [103]:

migrated = migrate_customer(
    customer_v1
)

print(
    json.dumps(
        migrated,
        indent=2
    )
)


{
  "schema_version": 2,
  "profile": {
    "full_name": "Ada Lovelace",
    "emails": [
      "ada@example.com"
    ]
  }
}


Versioning is one of the most important practices for long-lived JSON data.

Without a version field, future code has to guess which document shape it received.


## Problem 27 — Safe tagged decoding

Tagged JSON objects are useful, but we should be careful with untrusted data.

A dangerous design would allow the JSON to specify arbitrary Python modules or class names to import and instantiate.

That would give untrusted data too much control.

A safer design uses an explicit allowlist.


In [104]:

def decode_decimal(obj):
    return Decimal(
        obj["value"]
    )


def decode_uuid(obj):
    return UUID(
        obj["value"]
    )


SAFE_DECODERS = {
    "decimal": decode_decimal,
    "uuid": decode_uuid
}


In [105]:

def safe_object_hook(obj):
    tag = obj.get("__type__")

    if tag is None:
        return obj

    decoder = SAFE_DECODERS.get(tag)

    if decoder is None:
        return obj

    return decoder(obj)


Now only known tags are interpreted.


In [106]:

text = json.dumps({
    "price": {
        "__type__": "decimal",
        "value": "12.50"
    },
    "unknown": {
        "__type__": "SomeRandomClass",
        "value": "data"
    }
})

restored = json.loads(
    text,
    object_hook=safe_object_hook
)

print(restored)
print(
    type(restored["price"])
)
print(
    type(restored["unknown"])
)


{'price': Decimal('12.50'), 'unknown': {'__type__': 'SomeRandomClass', 'value': 'data'}}
<class 'decimal.Decimal'>
<class 'dict'>


The known decimal tag is decoded.

The unknown tag remains an ordinary dictionary.

No dynamic import or arbitrary object construction occurs.


## Problem 28 — Build a small production-style codec

Let us combine several lessons into a small reusable codec for invoices.

Requirements:

1. explicit document representation,
2. schema version,
3. strict JSON output,
4. deterministic key ordering,
5. UTF-8-friendly text,
6. duplicate-key rejection when loading,
7. semantic validation.


In [107]:

def invoice_to_versioned_document(
    invoice
):
    return {
        "schema_version": 1,
        "invoice": {
            "invoice_id": str(
                invoice.invoice_id
            ),
            "customer": (
                invoice.customer
            ),
            "amount": str(
                invoice.amount
            ),
            "issued_at": (
                invoice.issued_at
                .isoformat()
            )
        }
    }


In [108]:

def invoice_from_versioned_document(
    document
):
    if document.get(
        "schema_version"
    ) != 1:
        raise ValueError(
            "Unsupported schema version"
        )

    raw = document.get("invoice")

    if not isinstance(raw, dict):
        raise ValueError(
            "invoice must be an object"
        )

    return validated_invoice_from_document(
        raw
    )


In [109]:

class InvoiceJSONCodec:
    @staticmethod
    def dumps(invoice):
        document = (
            invoice_to_versioned_document(
                invoice
            )
        )

        return json.dumps(
            document,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False
        )

    @staticmethod
    def loads(text):
        document = json.loads(
            text,
            object_pairs_hook=(
                reject_duplicate_keys
            )
        )

        return (
            invoice_from_versioned_document(
                document
            )
        )


Let us test the codec.


In [110]:

invoice = Invoice(
    invoice_id=uuid4(),
    customer="Katherine Johnson",
    amount=Decimal("987.65"),
    issued_at=datetime(
        2026, 8, 7, 16, 15,
        tzinfo=timezone.utc
    )
)

text = InvoiceJSONCodec.dumps(
    invoice
)

print(text)


{"invoice":{"amount":"987.65","customer":"Katherine Johnson","invoice_id":"7163aa4a-eaff-4c9d-97b2-570e5eb54fdf","issued_at":"2026-08-07T16:15:00+00:00"},"schema_version":1}


In [111]:

restored = InvoiceJSONCodec.loads(
    text
)

print(restored)
print(restored == invoice)


Invoice(invoice_id=UUID('7163aa4a-eaff-4c9d-97b2-570e5eb54fdf'), customer='Katherine Johnson', amount=Decimal('987.65'), issued_at=datetime.datetime(2026, 8, 7, 16, 15, tzinfo=datetime.timezone.utc))
True


This is a much more reliable serialization boundary than simply calling `vars()` on arbitrary Python objects.

The representation is explicit, versioned and validated.


# More advanced practice problems

The following problems are intentionally shorter.

Try to solve each one before reading the solution.


## Problem 29 — Preserve `frozenset`

Design a tagged representation that preserves the distinction between:

```python
set([1, 2, 3])
frozenset([1, 2, 3])
```


### Solution


In [112]:

def encode_set_like(value):
    if isinstance(value, frozenset):
        return {
            "__type__": "frozenset",
            "items": sorted(value)
        }

    if isinstance(value, set):
        return {
            "__type__": "set",
            "items": sorted(value)
        }

    raise TypeError


In [113]:

def decode_set_like(obj):
    if obj.get("__type__") == "set":
        return set(obj["items"])

    if obj.get(
        "__type__"
    ) == "frozenset":
        return frozenset(
            obj["items"]
        )

    return obj


In [114]:

original = {
    "mutable": {1, 2, 3},
    "immutable": frozenset(
        [4, 5, 6]
    )
}

text = json.dumps(
    original,
    default=encode_set_like
)

restored = json.loads(
    text,
    object_hook=decode_set_like
)

print(restored)
print(
    type(restored["mutable"])
)
print(
    type(restored["immutable"])
)


{'mutable': {1, 2, 3}, 'immutable': frozenset({4, 5, 6})}
<class 'set'>
<class 'frozenset'>


## Problem 30 — Reject very large JSON text before parsing

When JSON comes from an untrusted source, it can be useful to enforce a maximum input size before decoding.

Write a helper with a byte limit.


### Solution


In [115]:

def loads_with_size_limit(
    text,
    max_bytes=1024
):
    size = len(
        text.encode("utf-8")
    )

    if size > max_bytes:
        raise ValueError(
            f"JSON input is {size} bytes; "
            f"limit is {max_bytes}"
        )

    return json.loads(text)


In [116]:

small = '{"a": 1}'

print(
    loads_with_size_limit(
        small,
        max_bytes=100
    )
)


{'a': 1}


In [117]:

large = json.dumps({
    "data": "x" * 1000
})

try:
    loads_with_size_limit(
        large,
        max_bytes=100
    )
except ValueError as ex:
    print(ex)


JSON input is 1012 bytes; limit is 100


## Problem 31 — Measure nesting depth after parsing

Deeply nested structures can be inconvenient or dangerous for application logic.

Write a function that rejects data deeper than a chosen level.


### Solution


In [118]:

def nesting_depth(value):
    if isinstance(value, dict):
        if not value:
            return 1

        return 1 + max(
            nesting_depth(v)
            for v in value.values()
        )

    if isinstance(value, list):
        if not value:
            return 1

        return 1 + max(
            nesting_depth(v)
            for v in value
        )

    return 0


In [119]:

def assert_max_depth(
    value,
    max_depth
):
    depth = nesting_depth(value)

    if depth > max_depth:
        raise ValueError(
            f"JSON nesting depth {depth} "
            f"exceeds limit {max_depth}"
        )


In [120]:

deep = {
    "a": {
        "b": {
            "c": {
                "d": 1
            }
        }
    }
}

print(
    nesting_depth(deep)
)

try:
    assert_max_depth(
        deep,
        max_depth=3
    )
except ValueError as ex:
    print(ex)


4
JSON nesting depth 4 exceeds limit 3


## Problem 32 — Redact secrets before JSON logging

Applications often log JSON-like structures.

We should not accidentally log passwords, tokens or secrets.

Write a recursive redaction function.


### Solution


In [121]:

SENSITIVE_KEYS = {
    "password",
    "token",
    "secret",
    "api_key"
}


In [122]:

def redact(value):
    if isinstance(value, dict):
        result = {}

        for key, item in value.items():
            if key.lower() in SENSITIVE_KEYS:
                result[key] = "***REDACTED***"
            else:
                result[key] = redact(item)

        return result

    if isinstance(value, list):
        return [
            redact(item)
            for item in value
        ]

    return value


In [123]:

request = {
    "username": "alice",
    "password": "super-secret",
    "headers": {
        "token": "abc123"
    },
    "items": [
        {
            "api_key": "key-1",
            "value": 10
        }
    ]
}

safe_request = redact(request)

print(
    json.dumps(
        safe_request,
        indent=2
    )
)


{
  "username": "alice",
  "password": "***REDACTED***",
  "headers": {
    "token": "***REDACTED***"
  },
  "items": [
    {
      "api_key": "***REDACTED***",
      "value": 10
    }
  ]
}


## Problem 33 — Test serialization behavior

Serialization code should be tested like any other code.

A useful test suite should cover:

- successful round trips,
- type restoration,
- invalid values,
- duplicate keys,
- strict numeric behavior,
- deterministic output.


### Solution


In [124]:

def test_decimal_round_trip():
    value = Decimal(
        "0.12345678901234567890"
    )

    text = json.dumps(
        {"value": value},
        default=decimal_default
    )

    restored = json.loads(
        text,
        object_hook=decimal_object_hook
    )

    assert restored["value"] == value
    assert isinstance(
        restored["value"],
        Decimal
    )


In [125]:

def test_duplicate_rejection():
    text = '{"x": 1, "x": 2}'

    try:
        json.loads(
            text,
            object_pairs_hook=(
                reject_duplicate_keys
            )
        )
    except ValueError:
        return

    raise AssertionError(
        "Duplicate key was accepted"
    )


In [126]:

def test_nan_rejection():
    try:
        json.dumps(
            {"x": float("nan")},
            allow_nan=False
        )
    except ValueError:
        return

    raise AssertionError(
        "NaN was accepted"
    )


In [127]:

def test_deterministic_output():
    left = {
        "b": 2,
        "a": 1
    }

    right = {
        "a": 1,
        "b": 2
    }

    assert (
        deterministic_json(left)
        ==
        deterministic_json(right)
    )


In [128]:

tests = [
    test_decimal_round_trip,
    test_duplicate_rejection,
    test_nan_rejection,
    test_deterministic_output
]

for test in tests:
    test()
    print(
        "PASS",
        test.__name__
    )


PASS test_decimal_round_trip
PASS test_duplicate_rejection
PASS test_nan_rejection
PASS test_deterministic_output


# Final review

We started with simple serialization problems and gradually moved toward application-level design.

The most important lessons are:

- JSON supports only a small type system.
- Python objects may serialize successfully while losing type information.
- Dictionary keys are especially important because JSON object keys are strings.
- `Decimal`, `UUID`, `datetime`, sets, enums and binary data need deliberate representations.
- `default=` helps customize encoding.
- `object_hook=` helps customize decoding.
- `parse_float=Decimal` is useful when receiving exact decimal values.
- tagged representations can preserve type information.
- tuple preservation usually requires preprocessing because tuples are already supported as JSON arrays.
- JSON parsing does not replace semantic validation.
- duplicate-key rejection can be useful for sensitive data.
- strict JSON should usually reject NaN and infinity.
- long-lived documents benefit from schema versions.
- event streams often work well with JSON Lines.
- cyclic object graphs should be represented using IDs or references.
- untrusted tagged data should be decoded through an allowlist, not arbitrary imports.
- deterministic JSON is useful for testing, hashing and caching.
- logging should redact secrets before serialization.


## Final thought

A good serializer does not simply ask:

> "How do I make Python stop raising `TypeError`?"

A better question is:

> "What JSON representation clearly expresses the meaning of this data, can be validated, can evolve over time, and can be reconstructed safely?"

That shift in thinking is what turns JSON serialization from a syntax exercise into software design.
